In [1]:
!module load python


The following have been reloaded with a version change:
  1) python/3.9.6-gc563 => python/3.13.9-ij92



In [2]:
pip install qepy qiskit qiskit-nature qiskit-algorithms numpy scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [3]:
!export W90_BIN="/home/am4655/q-e/bin/wannier90.x"
!export PW2W90_BIN="/home/am4655/q-e/bin/pw2wannier90.x"

In [17]:
# --- Minimal setup + run script (QEpy → hr.dat → FermionicOp → Qiskit VQE) ---

from pathlib import Path

# 1) EDIT ME: basic config
SCF_IN  = "./CH4.scf.in"     # QE SCF input
NSCF_IN = None    # QE NSCF input
WORKDIR = "work_w90"                  # output folder
SEED    = "CH4"

# Wannierization knobs
NUM_WANN  = 4
NUM_BANDS = None                   # or None
PROJECTIONS = ["c: s; p", "h: s"]
DIS_WIN  = (-8.0, 6.0)         # or None
DIS_FROZ = (-5.0, 0.5)         # or None
PW_OUTDIR = 'work'                 # set if your QE inputs use custom outdir

# Model + Qiskit knobs
KPOINT      = (0.0, 0.0, 0.0)    # Γ
SPINFUL     = False
U_ONSITE    = 0.0                # eV
NN_PAIRS    = None
MAPPER      = "jw"               # "jw" | "parity" | "bk"
RUN_VQE     = True
ANSATZ_LAYERS = 2
penalty      = 10.0
SHOTS         = None  # 0 → exact expectations; else e.g. 8192

In [18]:
# 2) Imports from your helpers
from wannierize_qepy import generate_hr_with_qepy
from ham_builder import ModelSpec, fermionic_from_hr, qubit_from_hr_penalized

# 3) Optional: Qiskit VQE bits
from qiskit.primitives import Estimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms.optimizers import COBYLA
import numpy as np

In [21]:
# --- Run Wannierization ---
print("[1/3] QEpy → Wannier90 …")
hr = generate_hr_with_qepy(
    scf_in=SCF_IN, nscf_in=NSCF_IN, out_dir=WORKDIR, seedname=SEED,
    num_wann=NUM_WANN, num_bands=NUM_BANDS,
    projections=PROJECTIONS, dis_win=DIS_WIN, dis_froz=DIS_FROZ,
    pw_outdir=PW_OUTDIR,
)
print(f"[ok] hr.dat: {hr}")

# --- Build molecular Hamiltonian (FermionicOp) ---
print("[2/3] Building FermionicOp …")
spec = ModelSpec(spinful=SPINFUL, U=U_ONSITE, nn_pairs=NN_PAIRS, unit_scale=1.0)
fop = fermionic_from_hr(hr, k=KPOINT, spec=spec)
print(f"[ok] spin-orbitals: {fop.num_spin_orbitals}")

# half-filling for nbasis spin-orbitals (2 per orbital):
electrons = 2 * (fop.num_spin_orbitals // 2)
# --- Map to qubits ---
print("[3/3] Mapping to qubits …")
qop, fop_pen = qubit_from_hr_penalized(hr_path=hr, k=KPOINT, spec=spec, n_target=electrons, penalty_coef=penalty, mapper=MAPPER,)
print(f"[ok] qubits={qop.num_qubits}  pauli_terms={len(qop)}")


[1/3] QEpy → Wannier90 …
[ok] hr.dat: /cache/home/am4655/qepy_eg/QCOMP/work_w90/CH4_hr.dat
[2/3] Building FermionicOp …
[ok] spin-orbitals: 8
[3/3] Mapping to qubits …
[ok] qubits=8  pauli_terms=93


In [22]:
# --- Optional: run a small VQE ---

if RUN_VQE:
    print("[VQE] Starting …")
    ansatz = EfficientSU2(qop.num_qubits, reps=ANSATZ_LAYERS, entanglement="linear")
    opt = COBYLA(maxiter=700)

    def cb(eval_count, params, value, meta):
        e = float(np.real_if_close(value))
        print(f"[VQE] iter={eval_count:03d}  E={e:.8f}")

    estimator = Estimator(options={"shots": None if SHOTS == 0 else SHOTS})
    vqe = VQE(estimator, ansatz, opt, callback=cb)
    res = vqe.compute_minimum_eigenvalue(qop)
    print("[VQE] final energy:", float(np.real_if_close(res.eigenvalue)))
else:
    print("[info] Skipping VQE (set RUN_VQE=True to execute)")

[VQE] Starting …
[VQE] iter=001  E=129.03265079
[VQE] iter=002  E=158.31440651
[VQE] iter=003  E=144.27455783
[VQE] iter=004  E=123.30435969
[VQE] iter=005  E=117.95799803
[VQE] iter=006  E=129.51532894
[VQE] iter=007  E=120.50934623
[VQE] iter=008  E=109.33017144
[VQE] iter=009  E=109.43032244
[VQE] iter=010  E=109.66919526
[VQE] iter=011  E=108.92622439
[VQE] iter=012  E=108.39084431
[VQE] iter=013  E=117.07734094
[VQE] iter=014  E=109.10433174
[VQE] iter=015  E=111.59410477
[VQE] iter=016  E=103.39105538
[VQE] iter=017  E=101.14021930
[VQE] iter=018  E=110.78706649
[VQE] iter=019  E=96.71074267
[VQE] iter=020  E=105.37427482


/tmp/ipykernel_64915/2117499897.py:12: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator(options={"shots": None if SHOTS == 0 else SHOTS})


[VQE] iter=021  E=102.21796555
[VQE] iter=022  E=81.76255195
[VQE] iter=023  E=80.01572481
[VQE] iter=024  E=86.91168462
[VQE] iter=025  E=78.46030869
[VQE] iter=026  E=78.79298122
[VQE] iter=027  E=80.04065486
[VQE] iter=028  E=76.49613280
[VQE] iter=029  E=117.91166171
[VQE] iter=030  E=72.46617349
[VQE] iter=031  E=76.06138842
[VQE] iter=032  E=68.22497074
[VQE] iter=033  E=85.33541610
[VQE] iter=034  E=88.47682702
[VQE] iter=035  E=86.50748176
[VQE] iter=036  E=79.92017784
[VQE] iter=037  E=68.52701437
[VQE] iter=038  E=74.95312811
[VQE] iter=039  E=66.16105870
[VQE] iter=040  E=72.59644045
[VQE] iter=041  E=68.08804916
[VQE] iter=042  E=66.15941070
[VQE] iter=043  E=66.14463425
[VQE] iter=044  E=65.98246194
[VQE] iter=045  E=65.89106317
[VQE] iter=046  E=66.03012042
[VQE] iter=047  E=65.87904007
[VQE] iter=048  E=65.78743392
[VQE] iter=049  E=65.92091504
[VQE] iter=050  E=42.50114153
[VQE] iter=051  E=52.84292486
[VQE] iter=052  E=40.52839203
[VQE] iter=053  E=44.23619772
[VQE] it

[VQE] iter=314  E=-16.96382454
[VQE] iter=315  E=-17.61935757
[VQE] iter=316  E=-17.55091379
[VQE] iter=317  E=-17.13459051
[VQE] iter=318  E=-17.39899738
[VQE] iter=319  E=-17.51352541
[VQE] iter=320  E=-17.55338326
[VQE] iter=321  E=-17.99710753
[VQE] iter=322  E=-17.97387430
[VQE] iter=323  E=-17.35316982
[VQE] iter=324  E=-17.72461306
[VQE] iter=325  E=-18.08517717
[VQE] iter=326  E=-17.92797656
[VQE] iter=327  E=-18.09737949
[VQE] iter=328  E=-18.21702320
[VQE] iter=329  E=-18.04144149
[VQE] iter=330  E=-17.91136047
[VQE] iter=331  E=-16.86518504
[VQE] iter=332  E=-17.84874207
[VQE] iter=333  E=-18.43621137
[VQE] iter=334  E=-18.55389019
[VQE] iter=335  E=-17.69726135
[VQE] iter=336  E=-18.74621401
[VQE] iter=337  E=-18.29144348
[VQE] iter=338  E=-18.96131904
[VQE] iter=339  E=-18.14619097
[VQE] iter=340  E=-18.80885364
[VQE] iter=341  E=-19.14700454
[VQE] iter=342  E=-19.20104680
[VQE] iter=343  E=-19.09891579
[VQE] iter=344  E=-19.03163987
[VQE] iter=345  E=-18.79564176
[VQE] it

[VQE] iter=580  E=-22.23905235
[VQE] iter=581  E=-22.17301985
[VQE] iter=582  E=-22.20129979
[VQE] iter=583  E=-22.23994594
[VQE] iter=584  E=-22.24600302
[VQE] iter=585  E=-22.26073255
[VQE] iter=586  E=-22.26193139
[VQE] iter=587  E=-22.12880034
[VQE] iter=588  E=-22.23915031
[VQE] iter=589  E=-22.31636989
[VQE] iter=590  E=-22.30436786
[VQE] iter=591  E=-22.38792497
[VQE] iter=592  E=-22.39811298
[VQE] iter=593  E=-22.46846666
[VQE] iter=594  E=-22.57613718
[VQE] iter=595  E=-22.58294922
[VQE] iter=596  E=-22.54744141
[VQE] iter=597  E=-22.53504867
[VQE] iter=598  E=-22.55099292
[VQE] iter=599  E=-22.56431814
[VQE] iter=600  E=-22.58388683
[VQE] iter=601  E=-22.54787146
[VQE] iter=602  E=-22.56730023
[VQE] iter=603  E=-22.63242272
[VQE] iter=604  E=-22.59678600
[VQE] iter=605  E=-22.58974698
[VQE] iter=606  E=-22.62007360
[VQE] iter=607  E=-22.62447704
[VQE] iter=608  E=-22.64491303
[VQE] iter=609  E=-22.61222163
[VQE] iter=610  E=-22.64842617
[VQE] iter=611  E=-22.65635685
[VQE] it